In [0]:

#assumption notes of the given code
#1.Spark creates the volumes/schemas before creating tables
#2.Creates a CSV from the list of enumerated tuples with an ID
#3 Make directories for orders and customers
#4.Create customer tables in json using a choice of 3 cities or nulls, with random signup/updated_at timestamps
#5.Create two customers files
#6.Create 3 tables of orders, each starting at different ID ranges


import random, json, datetime as dt
import os


catalog, schema, volume = "tech_assessment", "bean_counter", "landing"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
base = f"/Volumes/{catalog}/{schema}/{volume}"

random.seed(42)
stores = ["LDN01", "MAN01", "LDS01", "BRS01", "WEB01"]
products = [(f"P{i:03d}", n, c, p) for i, (n, c, p) in enumerate([
   ("Flat White", "Hot Drinks", 3.20), ("Latte", "Hot Drinks", 3.40),
   ("Cappuccino", "Hot Drinks", 3.30), ("Iced Mocha", "Cold Drinks", 4.10),
   ("Croissant", "Food", 2.50), ("Banana Bread", "Food", 2.80),
   ("Beans 1kg", "Retail", 14.00), ("Reusable Cup", "Retail", 9.50)], start=1)]

# Products (clean reference data)
with open(f"{base}/products.csv", "w") as f:
   f.write("product_id,product_name,category,unit_price\n")
   for p in products:
       f.write(",".join(map(str, p)) + "\n")

# Customers (JSON lines, batch 1 and a later batch with a new column)
def customers(start, n, extra=False):
   rows = []
   for i in range(start, start + n):
       r = {"customer_id": f"C{i:05d}", "email": f"user{i}@example.com",
            "city": random.choice(["London", "Manchester", "Leeds", "Bristol", None]),
            "signup_date": str(dt.date(2024, 1, 1) + dt.timedelta(days=random.randint(0, 600))),
            "updated_at": str(dt.datetime(2025, 1, 1) + dt.timedelta(hours=random.randint(0, 2000)))}
       if extra:
           r["marketing_opt_in"] = random.choice([True, False])
       rows.append(r)
   return rows

def write_jsonl(path, rows):
   with open(path, "w") as f:
       for r in rows:
           f.write(json.dumps(r) + "\n")


for d in ["orders", "customers"]:
   os.makedirs(f"{base}/{d}", exist_ok=True)

write_jsonl(f"{base}/customers/customers_batch1.json", customers(1, 500))
# batch 2: 100 updated customers (changes) + 100 new, plus a NEW column
write_jsonl(f"{base}/customers/customers_batch2.json", customers(401, 200, extra=True))

# Orders: 3 daily files, with duplicates, nulls, bad values, mixed date formats
def orders(day, n, start_id):
   rows = []
   for i in range(n):
       oid = start_id + i
       p = random.choice(products)
       qty = random.choice([1, 1, 1, 2, 3, -1, 0])          # bad quantities
       cust = random.choice([f"C{random.randint(1, 550):05d}", "", None])  # missing customers
       ts = dt.datetime.combine(day, dt.time(random.randint(7, 20), random.randint(0, 59)))
       fmt = random.choice(["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M"])    # mixed formats
       rows.append([f"O{oid:06d}", cust, random.choice(stores), p[0], qty,
                    p[3] if random.random() > 0.03 else "", ts.strftime(fmt)])
       if random.random() < 0.05:                              # duplicates
           rows.append(rows[-1])
   return rows

for n, day in enumerate([dt.date(2025, 3, 1), dt.date(2025, 3, 2), dt.date(2025, 3, 3)]):
   with open(f"{base}/orders/orders_{day}.csv", "w") as f:
       f.write("order_id,customer_id,store_id,product_id,quantity,unit_price,order_ts\n")
       for r in orders(day, 2000, n * 10000):
           f.write(",".join(map(str, r)) + "\n")

print("Landing files:", os.listdir(f"{base}/orders"), os.listdir(f"{base}/customers"))

In [0]:
#Bronze tables
# Although I prefare dataframes in Polars, pyspark SQL was used soley due to no prior expereince with pySpark
# A re-write using dataframes could uploaded soon if requested


from pyspark.sql import SparkSession, functions as f

catalog = "tech_assessment"
landing_path = f"/Volumes/{catalog}/bean_counter/landing"


for layer in ['bronze','silver','gold']:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{layer}")



order_list = ['/Volumes/tech_assessment/bean_counter/landing/orders/orders_2025-03-01.csv',
              '/Volumes/tech_assessment/bean_counter/landing/orders/orders_2025-03-02.csv',
              '/Volumes/tech_assessment/bean_counter/landing/orders/orders_2025-03-03.csv'
              ]

customer_files =['/Volumes/tech_assessment/bean_counter/landing/customers/customers_batch1.json',
                 '/Volumes/tech_assessment/bean_counter/landing/customers/customers_batch2.json']


###Bronze orders###
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.bronze_orders")

def insert_bronze_orders(file):
    spark.sql(f""" 
              COPY INTO {catalog}.bronze.bronze_orders
              FROM (
                  SELECT
                  *,
                  _metadata.file_path as source_file,
                  CURRENT_TIMESTAMP() as ingestion_ts
                FROM '{file}'
              )
              FILEFORMAT =CSV
              FORMAT_OPTIONS (
                  'header' = 'true',
                  'inferSchema' = 'false')
                  COPY_OPTIONS ('mergeSchema' = 'true')
              """)


for order_file in order_list:
    insert_bronze_orders(order_file)


#spark.sql("""select * from  tech_assessment.bronze.bronze_orders""").show()



###bronze customers####
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.bronze_customers")


def insert_bronze_customers(file):
    bronze_customers = spark.sql(f""" 
              COPY INTO {catalog}.bronze.bronze_customers
              FROM (
                  SELECT
                  *,
                  _metadata.file_path as source_file,
                  CURRENT_TIMESTAMP() as ingestion_ts
                FROM '{file}'
              )
              FILEFORMAT =JSON
              FORMAT_OPTIONS ('mergeSchema' = 'true')
              COPY_OPTIONS ('mergeSchema' = 'true')
              """)
    return bronze_customers
    


for cust_file in customer_files:
    insert_bronze_customers(cust_file)


spark.sql("""select * from  tech_assessment.bronze.bronze_customers""").show()


In [0]:

from pyspark.sql import SparkSession,functions as f

catalog = "tech_assessment"
product_file = '/Volumes/tech_assessment/bean_counter/landing/products.csv'

# products.csv as a temp view so SQL can join to it
spark.read.csv(product_file, header=True, inferSchema=True).createOrReplaceTempView("products")




### create silver orders table###
def silver_orders():
    orders = spark.sql(f"""
        WITH dates AS (
            SELECT
                CAST(d.order_id AS STRING) AS order_id,
                NULLIF(TRIM(d.customer_id), '') AS customer_id,           
                CAST(d.store_id AS STRING) AS store_id,
                CAST(d.product_id AS STRING) AS product_id,
                CAST(d.quantity AS INT) AS quantity,
                COALESCE(                                                 
                    CAST(NULLIF(TRIM(d.unit_price), '') AS NUMERIC(10,2)),
                    CAST(p.unit_price AS NUMERIC(10,2))
                ) AS unit_price,
                COALESCE(
                    TRY_TO_TIMESTAMP(d.order_ts, 'dd/MM/yyyy HH:mm'),
                    TRY_TO_TIMESTAMP(d.order_ts, 'yyyy-MM-dd HH:mm:ss')
                ) AS order_ts,
                d.source_file                                             
            FROM {catalog}.bronze.bronze_orders d
            LEFT JOIN products p
                ON p.product_id = d.product_id
        )

        SELECT
            order_id,
            customer_id,
            store_id,
            product_id,
            quantity,
            unit_price,
            order_ts,

            -- rejection reason
            CASE
                WHEN customer_id IS NULL AND quantity <= 0 THEN 'cust_id missing and quantity not positive'
                WHEN customer_id IS NULL THEN 'cust_id missing'
                WHEN quantity <= 0 THEN 'quantity not positive'
                ELSE NULL
            END AS rejection_reason,
            source_file,
            CURRENT_TIMESTAMP() AS ingestion_timestamp
        FROM dates

            """)
    

    orders = orders.dropDuplicates()

    silver_orders_df = (
        orders.filter(f.col('rejection_reason').isNull()).drop('rejection_reason')
    )
    silver_orders_quarantine_df = (
        orders.filter(f.col('rejection_reason').isNotNull())
    )

    silver_orders_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{catalog}.silver.silver_orders")

    silver_orders_quarantine_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{catalog}.silver.silver_orders_quarantine")


silver_orders()

###Create customers silver table###                                        
cust_table =f"{catalog}.silver.silver_customers"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {cust_table} (
        customer_id STRING,
        city STRING,
        email STRING,
        marketing_opt_in BOOLEAN,
        signup_date DATE,
        updated_at TIMESTAMP,
        source_file STRING,
        ingestion_timestamp TIMESTAMP
    )
""")



def silver_customers():
    cust_s= spark.sql(f"""
        MERGE INTO {cust_table} t
        USING (
            WITH pull AS (
                SELECT
                    NULLIF(TRIM(customer_id), '') AS customer_id,
                    CAST(city AS STRING) AS city,
                    NULLIF(TRIM(email), '') AS email,
                    CAST(marketing_opt_in AS BOOLEAN) AS marketing_opt_in,
                    CAST(signup_date AS DATE) AS signup_date,
                    CAST(updated_at AS TIMESTAMP) AS updated_at,
                    CAST(source_file AS STRING) AS source_file,
 
                    ROW_NUMBER() OVER (
                        PARTITION BY NULLIF(TRIM(customer_id), '')
                        ORDER BY CAST(updated_at AS TIMESTAMP) DESC, ingestion_ts DESC
                    ) AS rn
 
                FROM {catalog}.bronze.bronze_customers
            )
 
            SELECT
                customer_id,
                city,
                email,
                marketing_opt_in,
                signup_date,
                updated_at,
                source_file,
                CURRENT_TIMESTAMP() AS ingestion_timestamp
 
            FROM pull
            WHERE rn = 1
              AND customer_id IS NOT NULL
        ) s
        ON t.customer_id = s.customer_id
        WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    return cust_s

silver_cust = silver_customers()



#Run code twice to check duplciates arent being added
# spark.sql("""SELECT count(*)FROM tech_assessment.silver.silver_orders""").show()


#Show customers table
spark.sql(f"select * from {catalog}.silver.silver_customers").show()


In [0]:
#Gold Tables

product_file = '/Volumes/tech_assessment/bean_counter/landing/products.csv'

# products.csv as a temp view so SQL can join to it
spark.read.csv(product_file, header=True, inferSchema=True).createOrReplaceTempView("products")
catalog = "tech_assessment"


### create gold sales table
def gold_sales():
    gold_sales=spark.sql(f"""
    WITH cast_types as(
        SELECT
            order_id,
            customer_id,
            store_id,
            product_id,
            quantity::INT as quantity,
            unit_price::NUMERIC(10,2) as unit_price,
            order_ts,
            order_ts::DATE as order_date
        FROM {catalog}.silver.silver_orders
        )

    SELECT
        store_id,
        COUNT(DISTINCT order_id) AS orders,
        SUM(quantity * unit_price) AS rev,
        order_date
    FROM cast_types
    GROUP BY store_id, order_date
       ORDER BY store_id,order_date, rev, orders

    """)
    return gold_sales

gold_daily_sales = gold_sales()

gold_daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.gold_daily_sales")

#gold_daily_sales.show()


#Create gold_product_ranking table
def gold_products():
    gold_products=spark.sql(f"""
        WITH pull_orders  AS (
        SELECT
            product_id,
            SUM(unit_price::NUMERIC(10,2) * quantity::INT) as revenue
        FROM {catalog}.silver.silver_orders
            GROUP BY 1
        )

        SELECT
            o.product_id,
            o.revenue,
            p.product_name,
            p.category,
            DENSE_RANK() OVER (ORDER BY o.revenue DESC) AS best_revenue
        FROM pull_orders o
            LEFT JOIN products p
                ON p.product_id = d.product_id
            ORDER by best_revenue

            """)

    return gold_products


gold_product_ranking = gold_sales()


gold_product_ranking.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.gold_product_ranking")


gold_product_ranking.show()